In [ ]:
import os; os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == "notebooks" else None

# 01 - Explore the ITER hybrid ramp-up environment

`rl_tokamak.env.RampupEnv` wraps `gymtorax.IterHybridEnv`: a 150 s, 1 s-step control
task for the current ramp-up phase of an ITER-hybrid-scenario TORAX simulation. This
notebook inspects the raw Gym-TORAX observation/action spaces, runs a random and an
open-loop reference episode, and compares both to the published benchmark numbers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rl_tokamak.env import EnvConfig, RampupEnv
from rl_tokamak.controllers import OpenLoopController, RandomController
from rl_tokamak.evaluate import run_controller
from rl_tokamak.plotting import plot_episodes

Building the environment constructs the TORAX simulator (~10 s). The first episode
additionally pays a JAX compilation cost (~50 s); later episodes take ~17 s for the
full 151-step (t = 0..150 s) horizon.

In [ ]:
env = RampupEnv(EnvConfig())
print("wrapper observation_space:", env.observation_space)
print("wrapper action_space:", env.action_space)
print("horizon (steps):", env.horizon)

## Raw Gym-TORAX observation

`env.inner.observation_space` is a `gymnasium.spaces.Dict` with two sub-`Dict`s,
`"profiles"` (one `Box` per radial profile variable) and `"scalars"` (one `Box`
per scalar variable). `env.observation_space` is the flat, normalised vector the
wrapper builds from a subset of these (`EnvConfig.obs_set`, default `"profiles"`).

In [ ]:
obs_space = env.inner.observation_space
profiles = obs_space["profiles"].spaces
scalars = obs_space["scalars"].spaces
print(f"{len(profiles)} profile variables, {len(scalars)} scalar variables")
print()
for k in ("T_e", "q", "j_total"):
    if k in profiles:
        print(f"profiles[{k!r}].shape = {profiles[k].shape}")

total = sum(int(np.prod(s.shape)) for s in profiles.values())
total += sum(int(np.prod(s.shape)) for s in scalars.values())
print()
print("total flattened size of the raw Gym-TORAX Dict observation:", total)
print("flattened size of the wrapper's vector observation (env.observation_space):",
      env.observation_space.shape[0])

## Raw Gym-TORAX action space

The raw action is a `Dict` of three sub-actuators: `"Ip"` (plasma current, A),
`"NBI"` (power, Gaussian deposition location and width), `"ECRH"` (same three
fields). The wrapper compresses this to a 3-vector in `[-1, 1]` (`action_set="powers"`,
the default): `[Ip command, P_NBI, P_ECRH]`, with the NBI/ECRH deposition location
and width fixed at the Gym-TORAX reference values.

In [ ]:
print("env.inner.action_space:")
for name, space in env.inner.action_space.spaces.items():
    print(f"  {name}: {space}")
print()
print("env.action_space (wrapper):", env.action_space)

With the default `ip_mode="delta"`, `action[0]` is not a plasma-current set-point but a
*rate* command: it is multiplied by the 0.2 MA/s ramp-rate limit and added to the
previous I_p, so `action[0] = 1.0` means "ramp I_p up as fast as the actuator allows".

## Random and open-loop episodes

`RandomController` samples the raw Gym-TORAX action space uniformly (seeded).
`OpenLoopController` replays the fixed reference trajectory used to build the
benchmark's TORAX config (I_p 3 -> 12.5 MA linear ramp, heating on at t = 99 s).

In [ ]:
random_result = run_controller(env, RandomController(env.inner.action_space, seed=0))
print("random episode: benchmark_return =", random_result["benchmark_return"],
      " failed =", random_result["failed"], " steps =", random_result["steps"])

In [ ]:
ol_result = run_controller(env, OpenLoopController())
print("open-loop episode: benchmark_return =", ol_result["benchmark_return"],
      " failed =", ol_result["failed"], " steps =", ol_result["steps"])

In [ ]:
paper = {"random": -10.79, "open_loop": 3.40}
print(f"{'policy':<12}{'this run':>12}{'paper':>10}")
print(f"{'random':<12}{random_result['benchmark_return']:>12.2f}{paper['random']:>10.2f}")
print(f"{'open-loop':<12}{ol_result['benchmark_return']:>12.2f}{paper['open_loop']:>10.2f}")

In [ ]:
random_df = pd.DataFrame(random_result["log"])
ol_df = pd.DataFrame(ol_result["log"])
fig = plot_episodes({"random": random_df, "open-loop": ol_df}, title="Random vs open-loop reference")
fig

## Radial profiles of the reference run

The wrapper only keeps a handful of radial indices per profile (see
`rl_tokamak.env.PROFILE_IDX`). To see the full radial `T_e` and `q` profiles at a few
times, step `env.inner` directly with a fresh `OpenLoopController` and keep the raw
`obs["profiles"]["T_e"]` / `["q"]` arrays. The radial grid is not returned by Gym-TORAX;
`np.linspace(0, 1, len(profile))` recovers the normalised radius `rho_norm` each array
is sampled on.

In [ ]:
raw = env.inner
ctrl = OpenLoopController()
ctrl.reset()
obs, _ = raw.reset()

targets = {50, 100, 150}
snapshots = {}
t = 0
if t in targets:
    snapshots[t] = {"T_e": np.ravel(obs["profiles"]["T_e"]), "q": np.ravel(obs["profiles"]["q"])}

done = False
while not done:
    obs, r, term, trunc, info = raw.step(ctrl.act(obs))
    t += 1
    done = term or trunc
    if t in targets:
        snapshots[t] = {"T_e": np.ravel(obs["profiles"]["T_e"]), "q": np.ravel(obs["profiles"]["q"])}

print("captured snapshots at t =", sorted(snapshots))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for t, snap in sorted(snapshots.items()):
    rho_te = np.linspace(0, 1, len(snap["T_e"]))
    axes[0].plot(rho_te, snap["T_e"], label=f"t = {t} s")
    rho_q = np.linspace(0, 1, len(snap["q"]))
    axes[1].plot(rho_q, snap["q"], label=f"t = {t} s")
axes[0].set_xlabel("rho_norm")
axes[0].set_ylabel("T_e [keV]")
axes[0].legend(fontsize=8)
axes[1].set_xlabel("rho_norm")
axes[1].set_ylabel("q")
axes[1].axhline(1.0, color="k", lw=0.8, ls="--")
fig.tight_layout()
fig

## Reward components of the reference run

The benchmark reward is a sum of four terms, computed per step in
`rl_tokamak.env.benchmark_components` and logged under `r_fusion_gain`, `r_h98`,
`r_q_min`, `r_q95`. Summed over the episode they should reproduce `benchmark_return`.

In [ ]:
log_df = pd.DataFrame(ol_result["log"])
components = ["r_fusion_gain", "r_h98", "r_q_min", "r_q95"]
totals = log_df[components].sum()
print(totals)
print()
print("sum of components:", float(totals.sum()), " vs benchmark_return:", ol_result["benchmark_return"])

In [ ]:
env.close()